<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/13_marca_de_agua_efecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13. Efecto de la marca de agua

La marca de agua de Google Maps aparece solo en imágenes de Jamaica, y Jamaica tiene muchas más imágenes sin
paneles que el resto (ver `04_marca_de_agua`). Un modelo podría haber aprendido "marca → sin paneles" en vez de
mirar los paneles. El test va a ser de Kingston, con marca, así que conviene saberlo antes de elegir modelo.

Este notebook lo mide **sin reentrenar**: toma los modelos ya guardados en Drive, les cambia la imagen de
entrada y mira cuánto cambia la probabilidad (o el puntaje, en los YOLO). Hay dos pruebas, cada una con su
control (detalle en `pdi_tp/marca_de_agua.py`):

| Prueba | Grupo "con marca" | Control | Si el modelo usa la marca |
|---|---|---|---|
| **Tapar** | Imágenes de Jamaica con marca, con la franja de abajo (fila 222 en adelante) reemplazada por el reflejo de las filas de arriba | Lo mismo en imágenes de Jamaica sin marca | La probabilidad **sube** más que en el control |
| **Pegar** | Las 131 imágenes de val con paneles de las otras fuentes, con la franja de abajo de una imagen de Jamaica con marca | Las mismas imágenes, con la franja de una imagen de Jamaica sin marca | La probabilidad **baja** más que en el control |

El control separa el efecto de la marca del de cambiar esa franja (por ejemplo, tapar un panel que estaba abajo).
**Pegar es la prueba principal**: usa muchas más imágenes y es pareada. Tapar en val tiene solo 9 imágenes con
marca; en train hay 40, pero esas el modelo ya las vio.

**No entrena ni escribe nada:** abre los resultados en solo lectura (`entorno.preparar(solo_lectura=True)`).

**Cómo correrlo:** una sesión de Colab por familia, con GPU T4 (en CPU también anda, pero los YOLO tardan).
Elegir `FAMILIA` en la primera celda y *Ejecutar todas*. Si Colab pide reiniciar tras instalar dependencias,
reiniciar y volver a correr desde el principio.

In [ ]:
FAMILIA = "clasificadores"  # "clasificadores" (las 9 CNN) o "detectores" (los 4 YOLO): una sesión por familia

# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

assert FAMILIA in ("clasificadores", "detectores"), FAMILIA
requisitos = "requirements-training.txt" if FAMILIA == "clasificadores" else "requirements-detection.txt"
if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / requisitos)], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pdi_tp import comparacion, datos, entorno, marca_de_agua, resultados, split

rutas = entorno.preparar(solo_lectura=True)  # monta Drive, pero no deja crear ni guardar corridas

## 1. Qué imágenes se modifican

Se corre el detector de marca sobre las 900 imágenes (como en `04_marca_de_agua`) y se arman las pruebas. Los
donantes de la franja son siempre imágenes de Jamaica **sin paneles**, para no pegar paneles; se eligen al azar
con semilla fija.

In [ ]:
division = split.cargar_split(rutas)
conjunto = {p.name: nombre for nombre, lista in division.items() for p in lista}
filas = []
for imagen in datos.listar_imagenes(rutas):
    filas.append({"archivo": imagen.name, "split": conjunto.get(imagen.name), "jamaica": split.es_jamaica(imagen.name),
                  "con_paneles": int(datos.tiene_paneles(datos.label_de(rutas, imagen))),
                  "marca": marca_de_agua.detectar(datos.leer_imagen(imagen))["marca"]})
tabla = pd.DataFrame(filas)
pruebas = marca_de_agua.armar_pruebas(tabla, semilla=42)
pd.crosstab([pruebas["prueba"], pruebas["split"]], [pruebas["grupo"], pruebas["y_real"].map({1: "con paneles", 0: "sin paneles"})])

In [ ]:
nombres = sorted(set(pruebas["archivo"]) | set(pruebas["donante"]) - {""})
originales = {nombre: datos.leer_imagen(rutas.imagenes / nombre) for nombre in nombres}
modificadas = np.stack([marca_de_agua.modificar(fila, originales.__getitem__) for _, fila in pruebas.iterrows()])
archivos = sorted(set(pruebas["archivo"]))  # imágenes que se predicen también sin modificar
sin_modificar = np.stack([originales[nombre] for nombre in archivos])
print(f"{len(archivos)} imágenes originales y {len(modificadas)} modificadas por corrida")

# Ejemplos: original y modificada, uno por prueba y grupo.
ejemplos = pruebas[pruebas["split"] == "val"].groupby(["prueba", "grupo"]).head(1)
fig, axs = plt.subplots(2, len(ejemplos), figsize=(3.2 * len(ejemplos), 6.6))
for columna, (i, fila) in enumerate(ejemplos.iterrows()):
    axs[0, columna].imshow(originales[fila["archivo"]])
    axs[0, columna].set_title(f"{fila['prueba']} · {fila['grupo']}\noriginal", fontsize=9)
    axs[1, columna].imshow(modificadas[i])
    axs[1, columna].set_title("modificada" + (f"\n(franja de {fila['donante']})" if fila["donante"] else ""), fontsize=8)
    for fila_ejes in axs[:, columna]:
        fila_ejes.axhline(marca_de_agua.FILA_INICIO_MARCA, color="yellow", lw=0.8, ls="--")
        fila_ejes.set_axis_off()
plt.tight_layout()
plt.show()

## 2. Predicciones de cada modelo

Entran las corridas vigentes de la familia elegida (las mismas que compara `15_comparacion_validacion`). Cada modelo
predice las imágenes originales y las modificadas:

- **CNN:** `modelo.keras` (`mejor.keras` en ResNet50) recibe la imagen RGB en 0–255; el preprocesado de cada red está dentro del modelo.
- **YOLO:** `modelo.pt` con los parámetros de su receta (`imgsz`, `conf`, `iou`, `max_det`). El puntaje de la
  imagen es la máxima confianza de sus cajas (0 si no hay), igual que en los notebooks 08 y 09.

**Control de la carga:** las predicciones de las imágenes originales de val tienen que coincidir con las guardadas
en `predicciones_val.csv`. Si la diferencia pasa de `TOLERANCIA` (0.01), el modelo no se está cargando o
preprocesando igual que al entrenar: esa corrida queda afuera de la sección 3 y hay que revisarla.

In [ ]:
corridas = comparacion.corridas_vigentes(resultados.cargar_corridas(rutas))
familia = "clasificador" if FAMILIA == "clasificadores" else "detector"
corridas = corridas[corridas["vigente"] & (corridas["familia"] == familia)].reset_index(drop=True)
if corridas.empty:
    raise RuntimeError(f"No hay corridas vigentes de {FAMILIA} en Drive.")
corridas[["corrida", "modelo", "modo", "fecha", "umbral"]]

In [ ]:
if FAMILIA == "clasificadores":
    from tensorflow import keras

    def predecir(carpeta, config, imagenes):
        # ResNet50 guarda el mejor checkpoint como mejor.keras; las demás CNN, como modelo.keras.
        pesos = next(carpeta / n for n in ("modelo.keras", "mejor.keras") if (carpeta / n).is_file())
        keras.backend.clear_session()
        modelo = keras.models.load_model(pesos, compile=False)
        return modelo.predict(imagenes.astype("float32"), batch_size=32, verbose=0).ravel()
else:
    import json

    import torch
    from ultralytics import YOLO

    def predecir(carpeta, config, imagenes):
        # La receta se lee de la corrida: no todos los YOLO guardados se entrenaron con el mismo tamaño de entrada.
        receta = json.loads((carpeta / "config.json").read_text())["params"]["receta"]
        print(f"  imgsz={receta['imgsz']}, conf={receta['conf']}, iou={receta['iou']}, max_det={receta['max_det']}")
        modelo = YOLO(str(carpeta / "modelo.pt"))
        puntajes = []
        for inicio in range(0, len(imagenes), 32):
            # Ultralytics espera arrays en BGR, como los lee OpenCV.
            lote = [np.ascontiguousarray(imagen[:, :, ::-1]) for imagen in imagenes[inicio:inicio + 32]]
            salidas = modelo.predict(source=lote, imgsz=receta["imgsz"], conf=receta["conf"],
                                     iou=receta["iou"], max_det=receta["max_det"],
                                     device=0 if torch.cuda.is_available() else "cpu",
                                     augment=False, half=False, save=False, verbose=False)
            puntajes += [float(s.boxes.conf.max()) if len(s.boxes) else 0.0 for s in salidas]
        return np.array(puntajes)


TOLERANCIA = 0.01  # diferencia máxima aceptada contra predicciones_val.csv; si no, la corrida queda afuera
filas, chequeo = [], []
for _, corrida in corridas.iterrows():
    carpeta = resultados.ruta_corrida(rutas, corrida["modelo"], corrida["modo"])
    print("Prediciendo", corrida["corrida"], "...")
    original = dict(zip(archivos, predecir(carpeta, corrida, sin_modificar)))
    modificada = predecir(carpeta, corrida, modificadas)
    guardadas = pd.read_csv(carpeta / "predicciones_val.csv").set_index("archivo")["probabilidad"]
    comunes = [nombre for nombre in archivos if nombre in guardadas.index]
    diferencia = np.abs(np.array([original[n] for n in comunes]) - guardadas.loc[comunes].to_numpy())
    chequeo.append({"corrida": corrida["corrida"], "imagenes_val": len(comunes),
                    "diferencia_max": diferencia.max(), "diferencia_media": diferencia.mean()})
    filas.append(pruebas.assign(corrida=corrida["corrida"], umbral=corrida["umbral"],
                                original=pruebas["archivo"].map(original), modificada=modificada))
predicciones = pd.concat(filas, ignore_index=True)
chequeo = pd.DataFrame(chequeo)
chequeo["valida"] = chequeo["diferencia_max"] < TOLERANCIA
chequeo.style.format({"diferencia_max": "{:.4f}", "diferencia_media": "{:.5f}"})

## 3. Efecto de la marca

Para cada corrida y prueba:

- `cambio_con_marca` y `cambio_control`: cuánto cambia en promedio la probabilidad (modificada − original).
- `cambian_*`: cuántas imágenes cambian de predicción (cruzan el umbral de la corrida).
- `efecto` = cambio con marca − cambio en el control, con su intervalo de 95 % por bootstrap. En **pegar** se
  espera negativo y en **tapar** positivo si el modelo usa la marca.

**Cómo decidir:** hay evidencia de que un modelo usa la marca si el intervalo de `efecto` no incluye el 0 **y**
la marca cambia predicciones (no solo décimas de probabilidad que no cruzan el umbral). Mirar primero pegar en val.

In [ ]:
invalidas = chequeo.loc[~chequeo["valida"], "corrida"].tolist()
if invalidas:
    print("Quedan afuera porque la carga no reproduce sus predicciones guardadas (revisar antes de usarlas):", invalidas)
efecto = marca_de_agua.efecto(predicciones[~predicciones["corrida"].isin(invalidas)])
orden = {"pegar": 0, "tapar": 1}
efecto = efecto.sort_values(["prueba", "split", "efecto"], key=lambda c: c.map(orden) if c.name == "prueba" else c)
formato = {c: "{:+.3f}" for c in ("cambio_con_marca", "cambio_control", "efecto", "efecto_inf", "efecto_sup")}
efecto.style.format(formato).hide(axis="index")

In [ ]:
paneles = [("pegar", "val"), ("tapar", "val"), ("tapar", "train")]
fig, axs = plt.subplots(1, len(paneles), figsize=(6 * len(paneles), 0.45 * len(corridas) + 1.8), sharey=True)
etiquetas = corridas["corrida"].tolist()
for ax, (prueba, conjunto) in zip(axs, paneles):
    parte = efecto[(efecto["prueba"] == prueba) & (efecto["split"] == conjunto)].set_index("corrida").reindex(etiquetas)
    y = np.arange(len(etiquetas))
    ax.errorbar(parte["efecto"], y, xerr=[parte["efecto"] - parte["efecto_inf"], parte["efecto_sup"] - parte["efecto"]],
                fmt="o", capsize=3)
    ax.axvline(0, color="gray", lw=0.8)
    ax.set_yticks(y, etiquetas)
    n = parte["n_con_marca"].dropna()
    ax.set_title(f"{prueba} · {conjunto}" + (f" ({int(n.iloc[0])} imágenes con marca)" if len(n) else ""))
    ax.set_xlabel("efecto de la marca en la probabilidad")
axs[0].invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Qué hacer con el resultado

- **Ningún modelo muestra efecto** (intervalos que incluyen el 0 y casi ninguna predicción cambia): los modelos no
  usan la marca. No hace falta cambiar el entrenamiento; anotarlo en el issue #7 y en el informe, con esta tabla.
- **Algún modelo la usa:** neutralizarla al entrenar. La opción recomendada es pegar marcas falsas como
  augmentation en imágenes de las otras fuentes, para que la marca deje de indicar "Jamaica": no pierde
  información y no toca las cajas de detección. Tiene que aplicarse **antes** de la rotación de
  `aumentacion.google_maps`, que mueve la franja a cualquier borde. Después, volver a correr este notebook con el
  modelo nuevo para confirmar que el efecto desapareció.
- **Recordar:** las imágenes de val también eligieron el checkpoint. Esto no cambia la prueba (se compara el mismo
  modelo con y sin marca), pero el efecto en train es sobre imágenes que el modelo ya vio.